## در این مثال، یک فایل در خصوص نحوه بهره برداری از یک پمپ فراخوانی شده و با استفاده از مدل های NLP یک چت بات افلاین ایجاد و به سوالات تکنیسین پاسخ می دهد

برای ساخت یک چت‌بات کاملاً آفلاین که بتواند محتوای یک فایل را بخواند و به سوالات شما پاسخ دهد، بهترین راه استفاده از تکنولوژی RAG (Retrieval-Augmented Generation) است

### نصب کتابخانه های مورد نیاز

!pip install pymupdf sentence-transformers faiss-cpu transformers torch gpt4all

### فراخوانی کتابخانه های مورد نیاز

In [1]:
import os
from gpt4all import GPT4All
import pymupdf
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import re
import numpy as np

### فراخوانی مدل موردنیاز

مدل زبانی برای تعامل با کاربر استفاده می شود. تولید متن به صورت کلمه به کلمه بر اساس پرامپت ورودی. نقش تولید پاسخ نهایی را بر عهده دارد.

In [2]:
# دانلود مدل از https://huggingface.co/Qwen/Qwen2-7B-Instruct-GGUF/resolve/main/qwen2-7b-instruct-q2_k.gguf?download=true
# سپس قرار دادن در پوشه کاری
# این یک مدل زبانی با پشتیبانی فارسی است

#model = GPT4All("qwen2-7b-instruct-q2_k.gguf", model_path=".", allow_download=False) 

# یا استفاده از مدل زبانی دیگر
# https://huggingface.co/Yolozh/Qwen2-1.5B-Instruct-Q8_0-GGUF/resolve/main/qwen2-1.5b-instruct-q8_0.gguf?download=true
model = GPT4All("qwen2-1.5b-instruct-q8_0.gguf", model_path=".", allow_download=False) 

# https://huggingface.co/asedmammad/PersianMind-v1.0-GGUF/resolve/main/PersianMind-v1.0.q4_0.gguf?download=true
# model = GPT4All("persianmind-v1.0-q4_0.gguf", model_path=".", allow_download=False) 

# https://huggingface.co/MaziyarPanahi/LlaMAndement-7b-GGUF/resolve/main/LlaMAndement-7b.Q4_K_S.gguf?download=true
#model = GPT4All("LlaMAndement-7b.Q4_K_S.gguf", model_path=".", allow_download=False) 

# https://huggingface.co/jfer1015/Mistral-7B-Instruct-v0.3-Q4_K_M-GGUF/resolve/main/mistral-7b-instruct-v0.3-q4_k_m.gguf?download=true
#model = GPT4All("mistral-7b-instruct-v0.3-q4_k_m.gguf", model_path=".", allow_download=False) 

# https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf?download=true
#model = GPT4All("Phi-3-mini-4k-instruct-q4.gguf", model_path=".", allow_download=False) 

# تست مدل با زبان فارسی
with model.chat_session():
    print(model.generate("به فارسی سلام کن و بگو چه کمکی می توانی بکنی؟", max_tokens=3000))

سلام! من به شما درخواست دارم که برای اینترنت بازدید کنید. 

لطفاً مشخصات خود را (مثلا: سرعت، چند مورد، زمان و ...) بپرسید تا بتوانم بهتری در پاسخ داده شوم.




### فراخوانی سند و آماده سازی آن

سند فراخوانی شده و به بخش های کوچک تقسیم می شود و با مدل چندزبانه متن به یک بردار عددی با طول ثابت تبدیل می شود. این بردارها موقعیت معنایی متن را در فضای برداری نشان می دهند و ذخیره می شوند.  

In [3]:
# فراخوانی سند
doc = pymupdf.open("doc.pdf")
text = "\n".join(page.get_text() for page in doc)
    
# بخش بندی سند
chunk_size = 600    
overlap = 100

def clean_text(text):
    # حذف خطوط کشیده و نویز
    text = re.sub(r'[ـ]{2,}', '', text)
    # حذف اعداد تکراری مثل "1 1 1 1"
    text = re.sub(r'\b(\d+)(?:\s+\1)+\b', '', text)
    # حذف کاراکترهای عجیب
    text = re.sub(r'[□○●■]', '', text)
    # اصلاح فاصله‌ها
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# هنگام ساخت chunks:
chunks = [clean_text(text[i:i+chunk_size]) for i in range(0, len(text), chunk_size-overlap)]

# -------- ایجاد embeddings --------
#embed_model = SentenceTransformer("all-MiniLM-L6-v2/model")
embed_model = SentenceTransformer("multilingual-MiniLM-L12-v2/model")
embeddings = embed_model.encode(chunks)

# -------- ذخیره سازی در FAISS --------
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

# ذخیره ایندکس
faiss.write_index(index, "my_index.faiss")

with open("chunks_new.pkl", "wb") as f:
    pickle.dump(chunks, f)

In [4]:
# بارگذاری مجدد برای دفعات بعدی
index = faiss.read_index("my_index.faiss")
with open("chunks_new.pkl", "rb") as f:
    chunks = pickle.load(f)

### بررسی عملکرد بخش بندی و خوانش سند

In [5]:
test_query = "نحوه راه اندازی پمپ"
test_embedding = embed_model.encode([test_query])
distances, indices = index.search(np.array(test_embedding), 5)
for i, idx in enumerate(indices[0]):
    print(f"--- قطعه {i+1} (فاصله: {distances[0][i]:.2f}) ---")
    print(chunks[idx][:300])
    print("...")

--- قطعه 1 (فاصله: 9.80) ---
ntenance purposes. 2.6. DISASSEMBLY, REPAIR AND REASSEMBLY Before starting work on the pumpset, make sure it is disconnected from the mains and can not be switched on accidentally. Follow the safety precaution measures outlined in “safety instructions”. 2.6.1. Disassembly  Close all valves in the s
...
--- قطعه 2 (فاصله: 9.97) ---
7 2.2.3. Storage  If the pump is not to be installed and operated soon after arrival, store the pump in a clean, dry and frost-free place with moderate changes in ambient temperature.  If the pump has regreaseable bearings, pump extra grease on bearings to prevent moisture from entering around the
...
--- قطعه 3 (فاصله: 10.52) ---
ce to the local codes. - Any work on the pump should be only carried out when the unit has been brought to standstill. - Always disconnect the power to the motor and make sure not be switched on accidentally before working on the pump or removing the pump from installation. - Any work on the pump sh
.

با توجه به اینکه در خوانش سند با وجود استفاده از تابع تمیزکاری متن، کماکان می تواند اشکالاتی وجود داشته باشد، لذا یا باید در خوانش متن اصلاحات بیشتری انجام شود یا از مدل های زبانی قوی تری استفاده شود.

In [14]:
# --------تابع جستجو --------
def search(query, k=9):

    query_embedding = embed_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    results = [chunks[i] for i in indices[0]]

    return "\n".join(results)


# -------- حلقه گفتگو --------
print("گفتگو آغاز شد. برای خروج 'exit' را وارد کنید.")
with model.chat_session():
    while True:
        question = input("\nسوال خود را مطرح کنید: ")
        if question.lower() == "exit":
            break
        context = search(question)
        
        max_context_chars = 2000   # حدود 500 توکن
        if len(context) > max_context_chars:
            context = context[:max_context_chars] + "..."
        
        prompt = f"""Instructions: Answer the {question} on the following {context}"""
        
       # prompt = f"""دستورالعمل: فقط بر اساس متن زیر به سوال پاسخ بده. اگر جواب در متن نیست، بگو «اطلاعاتی موجود نیست». پاسخ باید روان به فارسی باشد. در انتها بنویس این اطلاعات از کدام بخش سند استخراج شده.
       # سوال: {question}        
       # متن: {context}
        

        
       # پاسخ (به صورت گام به گام):"""

       # prompt = f"""شما یک دستیار هستید. فقط بر اساس متن زیر پاسخ بده. اگر پاسخ دقیق پیدا نشد، بگو: «در متن موجود نیست»
       #        سوال: {question}                
       #         متن: {context}

       #        پاسخ کوتاه و دقیق به فارسی:
       #        """

        #  تولید خروجی
        output = model.generate(
            prompt,
            max_tokens=3000,
            temp=0.2,        # میزان خلاقیت مدل
            top_k=40,
            repeat_penalty=1.2
        )
        print(f"پاسخ: {output}")

گفتگو آغاز شد. برای خروج 'exit' را وارد کنید.



سوال خود را مطرح کنید:  oil replacement periods


پاسخ: The instructions state the following periods of changing oil:

1. For rated speed 2900 rpm: After operating hours of 1500.
2. For rated speed 960 - 1450 rpm: After operating hours of 3000.

For both speeds, it is recommended to use high-quality suitable lubricating oil such as SHELL TELLUS 46 cst for the first period and SHELL TELLUS 68 cst for the second. The instructions also mention that the bearing temperature should not exceed 50º C but never rise above 80º C.

For disassembly, repair, and reassembling of the pumpset before starting work on it:

1. Drain all grease from bearings with grease lubricated or remove oil if the pump is oil-lubricated.
2. Clean the bearings using a suitable cleaning fluid and relube them (not necessary for life-time grease lubricated bearings).
3. Install the pump in accordance with EN 60034-1, ensuring that it has proper protection against dust and moisture.

The instructions also mention:

1. The electrical motors should be built according to EN 


سوال خود را مطرح کنید:  exit
